In [1]:
import os

base = "/nfs/roberts/scratch/pi_sjf37/lm2445/FinAI_data_202604/FreeBSD CVS Archive/ncvs/src/bin"

In [2]:
for root, _, files in os.walk(base):
    for f in files:
        if f.endswith(",v"):
            print(os.path.join(root, f))
            break
    break

/nfs/roberts/scratch/pi_sjf37/lm2445/FinAI_data_202604/FreeBSD CVS Archive/ncvs/src/bin/Makefile,v


In [4]:
path = "/nfs/roberts/scratch/pi_sjf37/lm2445/FinAI_data_202604/FreeBSD CVS Archive/ncvs/src/bin/Makefile,v"

with open(path, "r", errors="ignore") as f:
    content = f.read()

print(content[:1000])   # first 1000 chars

head	1.31;
access;
symbols
	RELENG_8_4:1.31.0.2
	RELENG_9_1_0_RELEASE:1.30.2.1.4.2
	RELENG_9_1:1.30.2.1.0.4
	RELENG_9_1_BP:1.30.2.1
	RELENG_8_3_0_RELEASE:1.28.2.2.6.1
	RELENG_8_3:1.28.2.2.0.6
	RELENG_8_3_BP:1.28.2.2
	RELENG_9_0_0_RELEASE:1.30.2.1.2.1
	RELENG_9_0:1.30.2.1.0.2
	RELENG_9_0_BP:1.30.2.1
	RELENG_9:1.30.0.2
	RELENG_9_BP:1.30
	RELENG_7_4_0_RELEASE:1.26.2.2.4.1
	RELENG_8_2_0_RELEASE:1.28.2.2.4.1
	RELENG_7_4:1.26.2.2.0.4
	RELENG_7_4_BP:1.26.2.2
	RELENG_8_2:1.28.2.2.0.4
	RELENG_8_2_BP:1.28.2.2
	RELENG_8_1_0_RELEASE:1.28.2.2.2.1
	RELENG_8_1:1.28.2.2.0.2
	RELENG_8_1_BP:1.28.2.2
	RELENG_7_3_0_RELEASE:1.26.2.2.2.1
	RELENG_7_3:1.26.2.2.0.2
	RELENG_7_3_BP:1.26.2.2
	RELENG_8_0_0_RELEASE:1.28.2.1.2.1
	RELENG_8_0:1.28.2.1.0.2
	RELENG_8_0_BP:1.28.2.1
	RELENG_8:1.28.0.2
	RELENG_8_BP:1.28
	RELENG_7_2_0_RELEASE:1.26.2.1.4.1
	RELENG_7_2:1.26.2.1.0.4
	RELENG_7_2_BP:1.26.2.1
	RELENG_7_1_0_RELEASE:1.26.2.1.2.1
	RELENG_6_4_0_RELEASE:1.25.12.1
	RELENG_7_1:1.26.2.1.0.2
	RELENG_7_1_BP:1.26.2.1
	RELEN

In [5]:
import re

matches = re.findall(r"text\s*\n@(.+?)@", content, re.DOTALL)

print("Number of revisions:", len(matches))
print("\n===== SAMPLE =====\n")
print(matches[0][:500])


Number of revisions: 68

===== SAMPLE =====

#	From: 


In [6]:
for i in range(5):
    print(f"\n--- Revision {i} ---\n")
    print(matches[i][:300])


--- Revision 0 ---

#	From: 

--- Revision 1 ---

d1 58


--- Revision 2 ---

a0 57
#	From: 

--- Revision 3 ---

d2 1
a2 1
# $FreeBSD$


--- Revision 4 ---

@


1.30.2.2
log



In [35]:
path = "/nfs/roberts/scratch/pi_sjf37/lm2445/FinAI_data_202604/FreeBSD CVS Archive/ncvs/src/bin/ls/ls.c,v"
import re
import tiktoken

enc = tiktoken.get_encoding("cl100k_base")

with open(path, "r", errors="ignore") as f:
    content = f.read()

# revisions = re.split(r"\nrevision\s+", content)[1:]
revisions = re.split(r"\n\d+\.\d+(?:\.\d+)*\n", content)
revisions = revisions[1:]   # skip header
print("Total revisions:", len(revisions))

Total revisions: 306


In [36]:
revisions[2]

'date\t2012.11.08.23.45.19;\tauthor grog;\tstate Exp;\nbranches;\nnext\t1.96;\n'

In [37]:
import re
import tiktoken
from pathlib import Path

path = "/nfs/roberts/scratch/pi_sjf37/lm2445/FinAI_data_202604/FreeBSD CVS Archive/ncvs/src/bin/ls/ls.c,v"
SOURCE_NAME = "FreeBSD CVS Archive"

enc = tiktoken.get_encoding("cl100k_base")

def count_tokens(text):
    return len(enc.encode(text))

with open(path, "r", errors="ignore") as f:
    content = f.read()

# Split revisions while keeping the revision number
parts = re.split(r"\n(\d+\.\d+(?:\.\d+)*)\n", content)

# parts[0] = header
# then pairs: [rev_id, rev_block, rev_id, rev_block, ...]
samples = []

def extract_rcs_block(block, field_name):
    """
    Extract an RCS @...@ block after a field like:
    log
    @....@
    or
    text
    @....@
    
    Handles escaped @@ inside content.
    """
    m = re.search(rf"\n{field_name}\n@", block)
    if not m:
        return None

    i = m.end()   # position right after the opening @
    out = []

    while i < len(block):
        ch = block[i]

        if ch == "@":
            # Escaped @ inside content
            if i + 1 < len(block) and block[i + 1] == "@":
                out.append("@")
                i += 2
            else:
                # Closing @
                return "".join(out)
        else:
            out.append(ch)
            i += 1

    return None  # malformed block

for i in range(1, len(parts), 2):
    rev_id = parts[i]
    rev_block = parts[i + 1]

    # Extract year from date line like:
    # date 2012.11.08.23.45.19; author grog; state Exp; ...
    date_match = re.search(r"date\s+(\d{4})[./]", rev_block)
    if not date_match:
        continue

    year = int(date_match.group(1))

    text = extract_rcs_block(rev_block, "text")
    if text is None:
        continue

    text = text.strip()
    token_count = count_tokens(text)

    sample = {
        "Source": SOURCE_NAME,
        "Date": year,
        "Text": text,
        "Token_count": token_count,
    }
    samples.append(sample)

print("Total extracted samples:", len(samples))
print(samples[0] if samples else "No samples found")

Total extracted samples: 0
No samples found


In [38]:
import re
import json
import tiktoken

# -----------------------------
# Config
# -----------------------------
path = "/nfs/roberts/scratch/pi_sjf37/lm2445/FinAI_data_202604/FreeBSD CVS Archive/ncvs/src/bin/ls/ls.c,v"
OUTPUT_FILE = "freebsd_cvs_samples.jsonl"
SOURCE_NAME = "FreeBSD CVS Archive"

enc = tiktoken.get_encoding("cl100k_base")

def count_tokens(text):
    return len(enc.encode(text))


# -----------------------------
# Load file
# -----------------------------
with open(path, "r", errors="ignore") as f:
    content = f.read()


# -----------------------------
# Step 1: Build revision -> year map
# -----------------------------
rev2year = {}

meta_matches = re.finditer(
    r"\n(\d+\.\d+(?:\.\d+)*)\n"
    r"date\s+(\d{4})",
    content
)

for m in meta_matches:
    rev = m.group(1)
    year = int(m.group(2))
    rev2year[rev] = year

print("Total revisions with year:", len(rev2year))


# -----------------------------
# Step 2: Parse @...@ block (handles @@ escape)
# -----------------------------
def parse_cvs_text_block(s, start_idx):
    i = start_idx
    out = []

    while i < len(s):
        if s[i] == "@":
            # Escaped @@ → literal @
            if i + 1 < len(s) and s[i + 1] == "@":
                out.append("@")
                i += 2
            else:
                # Closing @
                return "".join(out)
        else:
            out.append(s[i])
            i += 1

    return None


# -----------------------------
# Step 3: Extract samples
# -----------------------------
samples = []

pattern = re.finditer(
    r"\n(\d+\.\d+(?:\.\d+)*)\nlog\n@",
    content
)

for m in pattern:
    rev = m.group(1)
    start = m.end()

    # --- Extract log (optional) ---
    log = parse_cvs_text_block(content, start)
    if log is None:
        continue

    # --- Find text block ---
    text_pos = content.find("\ntext\n@", start)
    if text_pos == -1:
        continue

    text_start = text_pos + len("\ntext\n@")
    text = parse_cvs_text_block(content, text_start)

    if not text:
        continue

    text = text.strip()

    if len(text) < 10:
        continue

    year = rev2year.get(rev)
    if year is None:
        continue

    samples.append({
        "Source": SOURCE_NAME,
        "Date": year,
        "Text": text,
        "Token_count": count_tokens(text)
    })


# -----------------------------
# Results
# -----------------------------
print("Total extracted samples:", len(samples))

if samples:
    print("\nSample example:")
    print(samples[0])


# -----------------------------
# Save to JSONL
# -----------------------------
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    for s in samples:
        f.write(json.dumps(s, ensure_ascii=False) + "\n")

print(f"\nSaved to {OUTPUT_FILE}")

Total revisions with year: 112
Total extracted samples: 95

Sample example:
{'Source': 'FreeBSD CVS Archive', 'Date': 2012, 'Text': '/*-\n * Copyright (c) 1989, 1993, 1994\n *\tThe Regents of the University of California.  All rights reserved.\n *\n * This code is derived from software contributed to Berkeley by\n * Michael Fischbein.\n *\n * Redistribution and use in source and binary forms, with or without\n * modification, are permitted provided that the following conditions\n * are met:\n * 1. Redistributions of source code must retain the above copyright\n *    notice, this list of conditions and the following disclaimer.\n * 2. Redistributions in binary form must reproduce the above copyright\n *    notice, this list of conditions and the following disclaimer in the\n *    documentation and/or other materials provided with the distribution.\n * 4. Neither the name of the University nor the names of its contributors\n *    may be used to endorse or promote products derived from th